### ***수정된 Prompt***

In [ ]:
initial_prompt="""### Scam indicators to look for:
The video should be considered potentially fraudulent if it contains **any** of the following characteristics:

1. **Monetary Requests:** Mentions of prepayment, fees, deposits, gift cards, or cryptocurrency transfers.
2. **Urgency or Scarcity:** Words or imagery implying “now”, “limited offer”, “only today”, “act fast”, etc.
3. **External Channel Redirection:** Mentions or visuals leading to Telegram, open chat, QR codes, or other messaging apps.
4. **Impersonation of Institutions:** Use of logos, certificates, uniforms, or documents mimicking banks, governments, media, or law enforcement.
5. **Exaggerated Profit or No-Risk Claims:** Unrealistic promises of high profit, “guaranteed returns”, or “zero risk”.
6. **Persistent Banners or Overlays:** On-screen phone numbers, repeated contact details, or fixed overlay banners.
7. **Investment/Profit/Transaction Evidence:** Screenshots or records of trading activity or fake profit statements.
8. **Visible Contact Info or Short URLs:** Display of contact numbers, wallet addresses, or shortened links (bit.ly, etc.).

Non-Scam indicators to look for:
The video should be considered non-fraudulent if it contains any of the following characteristics and lacks scam indicators:
1. **The video should be considered non-fraudulent if it primarily contains any of the following characteristics**: Informational or Educational Purpose: Provides explanations, tutorials, news, or reviews without requesting or suggesting monetary transactions.
2. **Neutral or Calm Tone**: Uses normal pacing and tone without urgency, countdowns, or “act now”–style language.
3. **No External Channel Promotion**: Does not display or mention Telegram, KakaoTalk, WhatsApp, QR codes, or external links for contact.
4. **Authentic or Recognized Branding**: Shows clear, legitimate organization or creator branding (e.g., school logos, news outlets, content creator watermarks) rather than fake institutions.
5. **No Financial Promise or Incentive**: Discusses topics unrelated to investment, profit, or risk-free income; focuses on education, lifestyle, or entertainment.
6. **Clean Visual Layout**: Lacks persistent on-screen banners, overlays, or contact information; video composition appears normal and uncluttered.
7. **No Transaction Evidence**: Does not display screenshots of trading apps, bank records, or profit statements; focuses on narrative or commentary instead.
8. **Absence of Contact or Payment Info**: Does not include phone numbers, wallet addresses, or shortened URLs; if any appear, they belong to verified, legitimate entities (e.g., university, media outlet).

### Output format (JSON):
After reviewing the video’s visual and textual content, respond **strictly** in the following JSON format:

{
  "is_scam": true | false,
  "confidence": float between 0.0 and 1.0,
  "risk": "low" | "mid" | "high",
  "evidence": ["concise evidence 1", "concise evidence 2", ...],
  "explanation": "A short paragraph explaining why this content is or is not a scam, referencing observed indicators."
}

---

### Guidelines:
- If **two or more scam indicators** appear, classify it as `is_scam: true`.
- Or else, classify it as `is_scam: false`.
- Set `"risk": "high"` for strong monetary solicitation or impersonation of institutions.
- Or else, set `"risk": "low"`
- Be cautious not to classify **educational or awareness videos about scams** as scams themselves — check for warning or explanatory tone.
- Respond in **English** only.
- Do not include extra commentary or formatting outside the JSON object
- Do not include extra commentary or formatting outside the JSON object
- Do not include extra commentary or formatting outside the JSON object
"""

### ***테스트 영상 및 라벨***
- 다운로드 받은 모든 테스트 영상에 대한 라벨

In [2]:
import os

abnormal_test_video_dir = '/home/ubuntu/cybercop/video_20251104/abnormal'
normal_test_video_dir = '/home/ubuntu/cybercop/video_20251104/normal'

video_paths = []
truth_labels = []

for root, dirs, files in os.walk(abnormal_test_video_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video_paths.append(file_path)
        truth_labels.append('[[Abnormal]]')

for root, dirs, files in os.walk(normal_test_video_dir):
    for f in files:
        file_path = os.path.join(root, f)
        video_paths.append(file_path)
        truth_labels.append('[[Normal]]')

In [3]:
for video_path, truth_label in zip(video_paths, truth_labels):
    print(video_path, truth_label)

/home/ubuntu/cybercop/video_20251104/abnormal/0017.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0013.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0009.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0000.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0032.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0026.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0008.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0030.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0002.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0037.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0003.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0010.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0012.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/0031.mp4 [[Abnormal]]
/home/ubuntu/cybercop/video_20251104/abnormal/00

### ***MiniCPM 모델 로드***
- Huggingface 코드스니펫 그대로 적용

In [4]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()

/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/ubuntu/anaconda3/envs/minicpm/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Loading checkpoint shards: 100%|███████

In [5]:
import math
import numpy as np
from PIL import Image
from moviepy.editor import VideoFileClip
import tempfile
import librosa
import ollama
import re
import traceback
import json
from json import JSONDecodeError

# -------------------------
# Extract video/audio chunks
# -------------------------
def get_video_chunk_content(video_path, flatten=True):
    video = VideoFileClip(video_path)
    print('video_duration:', video.duration)
    
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=True) as temp_audio_file:
        temp_audio_file_path = temp_audio_file.name
        video.audio.write_audiofile(temp_audio_file_path, codec="pcm_s16le", fps=16000)
        audio_np, sr = librosa.load(temp_audio_file_path, sr=16000, mono=True)

    num_units = math.ceil(video.duration)
    contents = []
    for i in range(num_units):
        frame = video.get_frame(i+1)
        image = Image.fromarray((frame).astype(np.uint8))
        audio = audio_np[sr*i:sr*(i+1)]
        if flatten:
            contents.extend(["<unit>", image, audio])
        else:
            contents.append(["<unit>", image, audio])
    return contents

# -------------------------
# MiniCPM inference with confidence
# -------------------------
def run_minicpm(video_path, prompt, model, tokenizer):
    contents = get_video_chunk_content(video_path)
    contents.append("<unit>")
    contents.append(prompt)
    
    sys_msg = model.get_sys_prompt(mode='omni', language='en')
    msg = {"role": "user", "content": contents}
    msgs = [sys_msg, msg]
    
    res = model.chat(
        msgs=msgs,
        tokenizer=tokenizer,
        sampling=True,
        temperature=1.0,
        max_new_tokens=4096,
        omni_input=True,
        use_tts_template=False,
        generate_audio=False,
        max_slice_nums=1,
        use_image_id=False,
        return_dict=True
    )
    
    # Assume MiniCPM can return a confidence score in res["confidence"] (or we can estimate)    
    output_text = res["text"].strip()    
    
    return output_text

# -------------------------
# gpt-oss prompt correction (strong version)
# -------------------------
def correct_prompt_with_llm(old_prompt, additional_prompt):
    instruction = f"""
You are optimizing a prompt for a multimodal LLM to detect scam videos.
MiniCPM received the following prompt but made an incorrect or low-confidence prediction.

Original prompt:
\"\"\"{old_prompt}\"\"\"

{additional_prompt}
"""
    llama_response = ollama.chat(
        model="gpt-oss:20b",
        messages=[{'role': 'user', 'content': instruction}],
        options = {'temperature': 1.0}
    )
    content = llama_response['message']['content']
    return content.strip()

# -------------------------
# Automatic loop with confidence threshold
# -------------------------
def auto_loop(video_path, initial_prompt, truth_label, model, tokenizer, max_iter=5, confidence_thresh=0.9):
    prompt = initial_prompt
    prompts = []
    #for i in range(max_iter):
    output = run_minicpm(video_path, prompt, model, tokenizer)
    # print(f"[MiniCPM] iteration {i+1} output: {output}")
    print(f"[MiniCPM] output: {output}")
    
    try:
        output_dict = json.loads(output)
    except JSONDecodeError:
        additional_prompt = f"There is JSONDecodeError on MiniCPM's output. Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence with correct JSON format"
        prompt = correct_prompt_with_llm(prompt, additional_prompt)

    prompts.append(output)

    # pred = "[[Abnormal]]" if output_dict['is_scam'] else "[[Normal]]"
    # conf = output_dict['confidence']
    
    # if pred == truth_label and conf >= confidence_thresh:
    #     print("[OK] Prediction matches truth with high confidence.")
    #     break
    # else:
    #     print("[!] Prediction incorrect or low confidence, correcting prompt...")
    #     additional_prompt = f'Revise the prompt so that MiniCPM will correctly classify such videos as {truth_label} scams with high confidence'
    #     # prompt = correct_prompt_with_llm(prompt, pred, conf, truth_label, additional_prompt)
    #     prompt = initial_prompt
    #     print(f"[!] New optimized prompt:\n{prompt}\n")
    return prompts

import time

final_prompts = []
for video_path, truth_label in zip(reversed(video_paths), reversed(truth_labels)):
# for video_path, truth_label in zip(video_paths, truth_labels):
    data = {'video_path': video_path, 'truth_label': truth_label}
    
    try:        
        print(video_path)
        start = time.perf_counter()
        prompts = auto_loop(
            video_path,
            initial_prompt,
            truth_label,
            model,
            tokenizer,
            max_iter=5,
            confidence_thresh=0.9
        )
        data['prompts'] = prompts    
        duration = time.perf_counter() - start
        data['duration'] = prompts    
        final_prompts.append(data)
    except:
        data['error'] = traceback.format_exc()
        traceback.print_exc()



/home/ubuntu/cybercop/video_20251104/normal/0025.mp4
video_duration: 18.34
MoviePy - Writing audio in /tmp/tmpt9vhx_n1.wav


MoviePy - Done.


The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.


[MiniCPM] output: {
  "is_scam": true,
  "confidence": 0.95,
  "risk": "high",
  "evidence": [
    "우버 택시 받tees",
    "오빠 택시 50% 허트 프로모션 코드 설치 중...",
    "지금 왜你还沒有返本呢？我已為你安装了50%的折扣！再等一會吧！"
  ],
  "explanation": "The video advertises a 'Uber Taxi' program with promises of significant discounts, prompting users to install promotional codes and offering immediate rewards such as an additional 50% discount for downloading the application. These strong signals align with typical scam tactics aimed at inducing quick actions without clear benefits."
}
/home/ubuntu/cybercop/video_20251104/normal/0011.mp4
video_duration: 34.62
MoviePy - Writing audio in /tmp/tmpi_qpbyji.wav


MoviePy - Done.


...

[MiniCPM] output: {
  "is_scam": true,
  "confidence": 0.85,
  "risk": "mid",
  "evidence": [
    "아이로 만든 '가짜 의사' 딥fake 광고 주의보",
    "이거 쓰고 aussi 업 잠시가 되는거에요",
    "아이로 만든 '가짜 의사' 영상 확산"
  ],
  "explanation": "The video features individuals who appear to be AI-generated, presenting themselves as medical professionals or cosmetic surgeons in advertisements for products and procedures. The consistent use of phrases like 'Deepfake advertisement caution,' the presence of people seemingly endorsing products with exaggerated claims about their effectiveness, and references to selling goods through online platforms suggest manipulative promotional tactics commonly associated with scams."
} 

This decision is based on multiple cues indicating a scam scenario: AI-generated visuals, suggestive endorsements without clinical context, product promotion through unclear methods (deepfake), emphasis on quick results or extreme promises which are red flags typical in deceptive advertising scenarios.
/h

MoviePy - Done.


[MiniCPM] output: {
  "is_scam": true,
  "confidence": 0.95,
  "risk": "high",
  "evidence": [
    "2025 MONSTA X CONNECT",
    "[수원티켓]蒙스타엑스 듬출 주변 지정석 인증",
    "AE매한作战1석 VIP석 FLOOR출 F2구역 6개"
  ],
  "explanation": "The video shows a seating chart for an event with mentions of special access and premium seats, indicating promotional or scam content. The presence of the text '[수원티켓]蒙스타엑스 듬출 주변 지정석 인증' suggests fraudulent claims about ticket authentication."
}
/home/ubuntu/cybercop/video_20251104/abnormal/0017.mp4
video_duration: 15.0
MoviePy - Writing audio in /tmp/tmp35a77oow.wav


MoviePy - Done.


[MiniCPM] output: {
  "is_scam": true,
  "confidence": 0.85,
  "risk": "mid",
  "evidence": [
    "保证胜利的交易，交易者可以利用这些信息来实现稳定的投资增长。",
    "投资回报率超过30%。"
  ],
  "explanation": "The video promotes financial investments with promises of high returns, suggesting a strong signal towards scam. The mention of guaranteed profit and specific percentages implies manipulative inducement without clear warnings or educational context."
}
